# 02 – SARIMA (Leitfaden Punkte 5–7)

Ziel: Ein saisonales ARIMA-Modell auf den Trainingsdaten identifizieren und anpassen, die 24 Testmonate prognostizieren und unter denselben Bedingungen wie Holt-Winters und die Seasonal-Naive-Baseline bewerten (gleicher Split, gleiche Metriken aus `src/evaluation.py`).

Regeln gegen Datenleckage:
- Die Abschnitte 1–6 verwenden **nur die Trainingsdaten**; die Testmenge wird dort gar nicht geladen (`train, _ = ...`).
- Das endgültige Modell wird in Abschnitt 6 **vor** der Prognose festgelegt.
- Die Testdaten werden erst in Abschnitt 7 einmalig verwendet.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_umsatz
from src.evaluation import train_test_split
from src import arima
from src.plots import speichern

s = load_umsatz(str(ROOT / "data/raw/45212-0004_de.csv"))
train, _ = train_test_split(s)
print(f"Training: {train.index[0]:%m/%Y}–{train.index[-1]:%m/%Y} ({len(train)} Monate)")

## 1. Varianzstabilisierung (Box-Cox)

ARIMA setzt eine (schwach) stationäre Reihe voraus, also auch eine konstante Varianz (Vorlesung 3, Folie 5). Ist die Varianz offensichtlich nicht konstant, wird sie mit einer Box-Cox-Transformation stabilisiert (Vorlesung 2, Folien 32–34).

**Entscheidungsregel (vor der Analyse festgelegt):** Transformiert wird nur, wenn die Standardabweichung je Jahr deutlich mit dem Jahresmittelwert steigt (Vorgehen wie in der Beispiel-Hausarbeit, Kap. 3). Das per Software geschätzte λ wird ergänzend berichtet.

In [ ]:
tab = arima.jahres_streuung(train)
r = tab["mittelwert"].corr(tab["std"])
print(f"Vollständige Jahre: {tab.index[0]}–{tab.index[-1]} ({len(tab)} Jahre)")
print(f"Korrelation Jahresmittelwert – Standardabweichung: r = {r:.2f}")
tab.round(2)

In [ ]:
fig = arima.plot_jahres_streuung(tab)
speichern(fig, "08_jahresstreuung")

In [ ]:
lam, ki = arima.boxcox_lambda(train)
print(f"λ (Maximum Likelihood) = {lam:.2f}, 95%-Konfidenzintervall [{ki[0]:.2f}; {ki[1]:.2f}]")

**Ergebnis und Entscheidung:**
- Zwischen Jahresmittelwert und jährlicher Standardabweichung besteht praktisch kein Zusammenhang (r = 0,06).
- Die größte Streuung tritt bei niedrigem Niveau auf (1999: Mittelwert 87,06, Standardabweichung 7,53), die kleinste bei vergleichsweise hohem Niveau (2022: 95,50 bzw. 4,52).
- Die Varianz ist damit nicht offensichtlich niveauabhängig. Gemäß der festgelegten Regel wird **keine Box-Cox-Transformation** angewendet; modelliert wird die Originalreihe.
- Das per Maximum Likelihood geschätzte λ = −1,76 (95%-KI [−2,84; −0,70]) dient nicht als Entscheidungsgrundlage: Das Verfahren unterstellt unabhängige, identisch verteilte Beobachtungen, was bei einer Reihe mit Trend und Saison nicht erfüllt ist.
- Konsequenz: Die Prognosen müssen nicht zurücktransformiert werden, und AIC-Vergleiche erfolgen auf der Originalskala.

## 2. Differenzierung

Ziel ist eine stationäre Reihe: konstanter Mittelwert, konstante Varianz, Kovarianz nur vom zeitlichen Abstand abhängig (Vorlesung 3, Folie 5). Differenzen werden mit dem Backshift-Operator geschrieben: $(1-B)\,Y_t = Y_t - Y_{t-1}$ und saisonal $(1-B^{12})\,Y_t = Y_t - Y_{t-12}$ (Vorlesung 3, Folien 37 und 41).

**Vorgehen (vor der Analyse festgelegt):**
1. Zuerst saisonal differenzieren ($D = 1$), danach regulär ($d = 1$) nur, falls noch ein Trend erkennbar ist; höchstens $d + D = 2$.
2. Hauptkriterium wie in Vorlesung 3, Folie 38: Mittelwert und Varianz wirken stabil, und die ACF fällt relativ schnell ab. Leitsatz: „So wenig differenzieren wie nötig“, da Überdifferenzierung künstliche Autokorrelation erzeugt.
3. Ergänzend (nicht Teil der Vorlesung): ADF-Test ($H_0$: nicht stationär) und KPSS-Test ($H_0$: stationär). Stationarität wird gestützt, wenn ADF ablehnt und KPSS nicht ablehnt. KPSS-p-Werte werden nur im Bereich 0,01–0,10 ausgegeben.
4. Die zusätzlich regulär differenzierte Reihe wird zum Vergleich mitberechnet, um eine Überdifferenzierung zu erkennen.

In [ ]:
reihen = {
    "Original $Y_t$": train,
    "$(1-B^{12})\\,Y_t$": arima.differenzieren(train, D=1),
    "$(1-B)(1-B^{12})\\,Y_t$": arima.differenzieren(train, d=1, D=1),
}
diff_tab = arima.differenzen_tabelle(reihen)
diff_tab.round(3)

In [ ]:
fig = arima.plot_differenzen(reihen)
speichern(fig, "09_differenzen");

**Ergebnis und Entscheidung:**
- **Originalreihe:** nicht stationär. Die ACF ist bei den saisonalen Lags hoch (Lag 12: 0,868) und fällt nur langsam ab; ADF lehnt nicht ab (p = 0,252), KPSS lehnt Stationarität ab (p ≤ 0,01).
- **Nach saisonaler Differenz $(1-B^{12})$:** Die Varianz sinkt von 52,59 auf 11,09, die Autokorrelation bei Lag 12 verschwindet (−0,005). Die Reihe schwankt um einen konstanten Mittelwert; die ACF liegt nach den ersten Lags überwiegend im Konfidenzband und zeigt keinen langsamen, linearen Abfall wie ein Random Walk (Vorlesung 3, Folie 12). ADF lehnt ab (p = 0,006), KPSS lehnt nicht ab (p ≥ 0,10): beide Tests stützen Stationarität.
- **Zusätzliche reguläre Differenz:** Die Autokorrelation bei Lag 1 wird stark negativ (−0,500) und die Varianz steigt auf 15,68. Das sind Anzeichen einer Überdifferenzierung (Vorlesung 3, Folie 38). Die Tests erkennen dies nicht (ADF p = 0,000), daher sind Grafik und ACF das Hauptkriterium.
- **Entscheidung:** $D = 1$, $d = 0$. Da $d + D = 1$, wird ein konstanter Term in das Modell aufgenommen und seine Signifikanz in Abschnitt 5 geprüft.
- Die verbleibende kurzfristige Autokorrelation der saisonal differenzierten Reihe wird in Abschnitt 3 über AR- und MA-Terme modelliert.

## 3. ACF und PACF der saisonal differenzierten Reihe

Grundlage ist $(1-B^{12})\,Y_t$ aus Abschnitt 2. Faustregeln aus Vorlesung 3: Die Ordnung $p$ ist der letzte signifikante Lag der PACF (Folie 26), die Ordnung $q$ der letzte signifikante Lag der ACF (Folie 30). Für den saisonalen Teil werden dieselben Regeln auf die Lags 12, 24 und 36 angewendet ($P$ aus der PACF, $Q$ aus der ACF). „Signifikant“ heißt: außerhalb des 95%-Bands.

In [ ]:
x = arima.differenzieren(train, D=1)
fig = arima.plot_acf_pacf(x, titel="$(1-B^{12})\\,Y_t$")
speichern(fig, "10_acf_pacf");

In [ ]:
acf_tab = arima.acf_pacf_tabelle(x)
print("Signifikante ACF-Lags: ", list(acf_tab.index[acf_tab["ACF signifikant"]]))
print("Signifikante PACF-Lags:", list(acf_tab.index[acf_tab["PACF signifikant"]]))
acf_tab.loc[[1, 2, 3, 4, 5, 6, 12, 24, 36]].round(3)

**Ergebnis: manuelles Modell**
- **Nicht-saisonal:** Die ACF ist bei fast allen Lags von 1 bis 11 signifikant und fällt allmählich ab, ohne abzubrechen → kein reiner MA-Prozess, $q = 0$. Die PACF ist bei Lag 1, 2 und 3 deutlich signifikant (0,293; 0,227; 0,422), bei Lag 4 nicht (0,097); Lag 5 überschreitet das Band nur knapp (0,139) und wird als Einzelwert nicht modelliert → $p = 3$ (Folie 26; kleinere Ordnungen bevorzugt, Folie 33).
- **Saisonal:** Die ACF ist bei Lag 12, 24 und 36 nicht signifikant → $Q = 0$. Die PACF ist bei Lag 12 und 24 signifikant (−0,364; −0,241), bei Lag 36 nicht (−0,055) → $P = 2$.
- **Manuelles Modell:** SARIMA(3,0,0)(2,1,0)$_{12}$ mit Konstante.
- **Einschränkungen:** Einzelne signifikante Werte bei höheren Lags (ACF: 14, 30; PACF: 14, 15, 16, 26, 34, 35) werden nicht modelliert; bei 36 Lags und 5 % Niveau sind je Funktion etwa 36 · 0,05 ≈ 1,8 zufällige Überschreitungen zu erwarten. Die abfallenden negativen PACF-Werte bei Lag 12 und 24 sind auch mit einem saisonalen MA-Term ($Q = 1$) vereinbar. Da die Bestimmung allein aus ACF und PACF schwierig ist (Folie 31), entscheidet in Abschnitt 4 die AIC-Suche über $P, Q \in \{0, 1, 2\}$.
- Das manuelle Modell liegt mit $p = 3$ außerhalb des Suchgitters ($p \le 2$) und wird in Abschnitt 4 zusätzlich geschätzt und per AIC verglichen (gleiches $d$, $D$ und gleiche Konstante).

## 4. Modellauswahl per AIC (Software)

Da die Bestimmung allein aus ACF und PACF schwierig ist (Vorlesung 3, Folie 31), werden die Ordnungen zusätzlich per Software gewählt (Folie 33). Kriterium ist das AIC (nicht Teil der Vorlesung):

$$\mathrm{AIC} = -2 \ln L + 2k$$

mit der maximierten Likelihood $L$ und der Anzahl geschätzter Parameter $k$ (einschließlich Konstante und $\sigma^2$). Kleiner ist besser; zusätzliche Parameter werden bestraft.

**Vorgehen (vor der Analyse festgelegt):**
- Gitter: $p, q, P, Q \in \{0, 1, 2\}$ (81 Modelle), fest: $d = 0$, $D = 1$, mit Konstante. Das AIC ist nur bei gleichem $d$, $D$ und gleicher Konstante vergleichbar.
- Das manuelle Modell aus Abschnitt 3 wird zusätzlich geschätzt.
- Kandidaten: das manuelle Modell, das Modell mit dem kleinsten AIC und das einfachste Modell (wenigste Parameter) mit ΔAIC < 2 zum besten Modell (gängige Faustregel; „eher kleinere Werte“, Folie 33).
- Die endgültige Wahl erfolgt in Abschnitt 5 nach der Residuendiagnose.

In [ ]:
import statsmodels
print("statsmodels", statsmodels.__version__)

gitter = arima.aic_gitter(train, d=0, D=1)   # dauert 1–3 Minuten
print(f"Geschätzte Modelle: {len(gitter)}, davon nicht konvergiert: {(~gitter['konvergiert']).sum()}")
gitter.head(10).round(2)

In [ ]:
manuell = arima.anpassen(train, (3, 0, 0), (2, 1, 0))
print(f"Manuelles Modell {arima.modell_name((3, 0, 0), (2, 1, 0))}: "
      f"AIC = {manuell.aic:.2f}, Parameter = {len(manuell.params)}, "
      f"ΔAIC zum besten Gittermodell = {manuell.aic - gitter['AIC'].min():.2f}, "
      f"konvergiert: {manuell.mle_retvals['converged']}")

In [ ]:
bester = gitter.iloc[0]
einfachster = gitter[gitter["ΔAIC"] < 2].sort_values(["Parameter", "AIC"]).iloc[0]
print(f"Bestes Modell (AIC):            {bester['Modell']} – {bester['Parameter']} Parameter, AIC = {bester['AIC']:.2f}")
print(f"Einfachstes Modell mit ΔAIC < 2: {einfachster['Modell']} – {einfachster['Parameter']} Parameter, ΔAIC = {einfachster['ΔAIC']:.2f}")